# Week 8-F: Query-Conditioned Latent Adapter (QCLA)
## Adapting Compressed Passage Embeddings Prior to Generator Conditioning

### Scientific Motivation:
Week 7-E proved that static SFR embeddings retain $+0.0106$ F1 utility over textless zeroing, but fail to deliver query-specific facts because passage embeddings are computed offline without knowledge of the query.

QCLA dynamically transforms latent representations using the query vector $e_q$:
$$\Delta e_D = W_2 \sigma(W_1 [e_D, e_q, e_D \odot e_q, |e_D - e_q|])$$
$$g = \sigma(W_g [e_D, e_q])$$
$$e'_D = e_D + g \odot \Delta e_D$$

### Models Compared:
- **QCLA-0**: Frozen SFR baseline (Identity mapping, 0 new parameters).
- **QCLA-1**: Query-conditioned bottleneck adapter with learned gate.
- **QCLA-2**: Query + Draft-conditioned adapter (incorporating draft answer representation $e_y$).


In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.week8.qcla import build_qcla_adapter

print("Initializing QCLA adapters...")
qcla0 = build_qcla_adapter("qcla_0")
qcla1 = build_qcla_adapter("qcla_1", dim=4096, hidden_dim=2048)
qcla2 = build_qcla_adapter("qcla_2", dim=4096, hidden_dim=2048)

p0 = sum(p.numel() for p in qcla0.parameters() if p.requires_grad)
p1 = sum(p.numel() for p in qcla1.parameters() if p.requires_grad)
p2 = sum(p.numel() for p in qcla2.parameters() if p.requires_grad)

print(f"QCLA-0 Trainable Parameters: {p0:,}")
print(f"QCLA-1 Trainable Parameters: {p1:,}")
print(f"QCLA-2 Trainable Parameters: {p2:,}")


In [ ]:
# Forward Pass Verification & Gating Behavior
batch_size = 2
e_doc = torch.randn(batch_size, 4096)
e_query = torch.randn(batch_size, 4096)
e_draft = torch.randn(batch_size, 4096)

# Forward QCLA-0 (Identity)
out_0 = qcla0(e_doc)
assert torch.allclose(out_0, e_doc), "QCLA-0 must be strict identity!"

# Forward QCLA-1
out_1 = qcla1(e_doc, e_query)
assert out_1.shape == (batch_size, 4096)

# Forward QCLA-2
out_2 = qcla2(e_doc, e_query, e_draft)
assert out_2.shape == (batch_size, 4096)

print("Forward pass shape and numerical consistency verified successfully!")


In [ ]:
# Inspect Pre-trained / Checkpointed Performance
ckpt_dir = REPO_ROOT / "results/week8/qcla_checkpoints"
print(f"Checkpoints will be saved to: {ckpt_dir}")
print("To run training over FIT set, execute:")
print("  python scripts/week8/run_w8f_qcla.py --adapter qcla_1 --epochs 3")
